# Operational extract quality check

Exploratory validation of the four supplied CSV extracts. The notebook never rewrites the raw files; production checks belong in backend code once the rules are agreed.

In [6]:
from pathlib import Path

import pandas as pd

DATA_DIR = next(path for path in (Path('data'), Path('../data')) if path.exists()).resolve()
FILES = {
    'trades': 'trades.csv',
    'market_data': 'market_data.csv',
    'risk_sensitivities': 'risk_sensitivities.csv',
    'fx_rates': 'fx_rates.csv',
}
frames = {name: pd.read_csv(DATA_DIR / filename) for name, filename in FILES.items()}

pd.DataFrame(
    {
        'rows': {name: len(frame) for name, frame in frames.items()},
        'columns': {name: len(frame.columns) for name, frame in frames.items()},
        'missing_cells': {name: int(frame.isna().sum().sum()) for name, frame in frames.items()},
    }
)

,rows,columns,missing_cells
trades,41,20,16
market_data,648,14,1992
risk_sensitivities,85,11,85
fx_rates,168,6,0


In [7]:
EXPECTED_COLUMNS = {
    'trades': {
        'trade_id', 'book_id', 'trader_id', 'trade_date', 'settle_date',
        'asset_class', 'product_type', 'instrument_id', 'currency',
        'notional', 'quantity', 'trade_price', 'direction', 'status',
        'maturity_date',
    },
    'market_data': {
        'date', 'instrument_id', 'asset_class', 'price', 'yield_pct',
        'spread_bps', 'implied_vol_pct', 'px_bid', 'px_ask', 'px_mid',
        'price_type', 'source', 'last_update_utc',
    },
    'risk_sensitivities': {
        'as_of_date', 'trade_id', 'book_id', 'instrument_id',
        'risk_metric', 'value', 'ccy', 'value_usd', 'unit',
        'computation_timestamp',
    },
    'fx_rates': {
        'date', 'ccy_pair', 'base_ccy', 'quote_ccy', 'spot_rate', 'source',
    },
}


schema_check = pd.DataFrame(
    [
        {
            'dataset': name,
            'missing_columns': sorted(expected - set(frames[name].columns)),
            'unexpected_columns': sorted(set(frames[name].columns) - expected),
        }
        for name, expected in EXPECTED_COLUMNS.items()
    ]
)
schema_check

,dataset,missing_columns,unexpected_columns
0,trades,[],"[bloomberg_id, counterparty_id, counterparty_n..."
1,market_data,[],[instrument_description]
2,risk_sensitivities,[],[notes]
3,fx_rates,[],[]


In [8]:
trades = frames['trades'].copy()
market = frames['market_data'].copy()
risk = frames['risk_sensitivities'].copy()
fx = frames['fx_rates'].copy()
issues = []

def add_issue(severity, dataset, code, count, detail):
    if count:
        issues.append({
            'severity': severity,
            'dataset': dataset,
            'code': code,
            'count': int(count),
            'detail': detail,
        })

duplicate_trades = trades.duplicated('trade_id', keep=False)
add_issue('ERROR', 'trades', 'DUPLICATE_TRADE_ID', duplicate_trades.sum(), 'Duplicate trade identifiers')

iso_trade_dates = pd.to_datetime(trades['trade_date'], format='%Y-%m-%d', errors='coerce')
us_trade_dates = pd.to_datetime(trades['trade_date'], format='%m/%d/%Y', errors='coerce')
trades['trade_date_parsed'] = iso_trade_dates.fillna(us_trade_dates)
add_issue('WARNING', 'trades', 'NON_ISO_TRADE_DATE', (iso_trade_dates.isna() & us_trade_dates.notna()).sum(), 'US-formatted dates normalized')
add_issue('ERROR', 'trades', 'INVALID_TRADE_DATE', trades['trade_date_parsed'].isna().sum(), 'Unparseable trade dates')

add_issue('ERROR', 'market_data', 'DUPLICATE_MARKET_KEY', market.duplicated(['date', 'instrument_id']).sum(), 'Duplicate date/instrument rows')
add_issue('ERROR', 'risk_sensitivities', 'DUPLICATE_RISK_KEY', risk.duplicated(['trade_id', 'risk_metric']).sum(), 'Duplicate trade/metric rows')
add_issue('ERROR', 'fx_rates', 'DUPLICATE_FX_KEY', fx.duplicated(['date', 'ccy_pair']).sum(), 'Duplicate date/pair rows')

trade_ids = set(trades['trade_id'])
risk_ids = set(risk['trade_id'])
add_issue('ERROR', 'risk_sensitivities', 'ORPHAN_RISK', len(risk_ids - trade_ids), 'Risk rows without a matching trade')
add_issue('WARNING', 'trades', 'MISSING_RISK', len(trade_ids - risk_ids), 'Trades without risk sensitivities')

non_fx_instruments = set(trades.loc[trades['asset_class'] != 'FX', 'instrument_id'])
missing_market = sorted(non_fx_instruments - set(market['instrument_id']))
add_issue('ERROR', 'market_data', 'MISSING_MARKET_DATA', len(missing_market), f'Missing instruments: {missing_market}')

for column in ('px_bid', 'px_mid', 'px_ask'):
    market[column] = pd.to_numeric(market[column], errors='coerce')
bad_quotes = market[['px_bid', 'px_mid', 'px_ask']].notna().all(axis=1) & ~((market['px_bid'] <= market['px_mid']) & (market['px_mid'] <= market['px_ask']))
add_issue('ERROR', 'market_data', 'INVALID_BID_MID_ASK', bad_quotes.sum(), 'Expected bid <= mid <= ask')

fx['spot_rate'] = pd.to_numeric(fx['spot_rate'], errors='coerce')
add_issue('ERROR', 'fx_rates', 'INVALID_SPOT_RATE', (fx['spot_rate'].isna() | (fx['spot_rate'] <= 0)).sum(), 'Spot rates must be positive numbers')
add_issue('ERROR', 'fx_rates', 'INVALID_CCY_PAIR', (fx['ccy_pair'] != fx['base_ccy'] + fx['quote_ccy']).sum(), 'Pair must equal base_ccy + quote_ccy')

pd.DataFrame(issues).sort_values(['severity', 'dataset', 'code']).reset_index(drop=True)

,severity,dataset,code,count,detail
0,ERROR,market_data,MISSING_MARKET_DATA,1,Missing instruments: ['HKLAND-3.875-2029']
1,ERROR,trades,DUPLICATE_TRADE_ID,2,Duplicate trade identifiers
2,WARNING,trades,NON_ISO_TRADE_DATE,2,US-formatted dates normalized


In [9]:
pd.DataFrame(
    [
        {'dataset': 'trades', 'latest_date': trades['trade_date_parsed'].max()},
        {'dataset': 'market_data', 'latest_date': pd.to_datetime(market['date'], errors='coerce').max()},
        {'dataset': 'risk_sensitivities', 'latest_date': pd.to_datetime(risk['as_of_date'], errors='coerce').max()},
        {'dataset': 'fx_rates', 'latest_date': pd.to_datetime(fx['date'], errors='coerce').max()},
    ]
)

,dataset,latest_date
0,trades,2026-07-31
1,market_data,2026-08-05
2,risk_sensitivities,2026-08-05
3,fx_rates,2026-08-05


## Boundary

This notebook is an exploratory aid, not the application data layer. Confirmed rules should move into a small backend validation module so API results do not depend on notebook execution. The notebook can then be removed without affecting the application.